# 셋팅 #

필요한 설정을 위해 이 셀을 실행하세요!


In [ ]:
import kagglehub
kagglehub.login()


In [ ]:
store_sales_time_series_forecasting_path = kagglehub.competition_download('store-sales-time-series-forecasting')
ryanholbrook_ts_course_data_path = kagglehub.dataset_download('ryanholbrook/ts-course-data')

print('Data source import complete.')


In [ ]:
import os
from pathlib import Path

# Define the target directory where learntools expects to find the data
input_base_path = Path('/input')
input_base_path.mkdir(parents=True, exist_ok=True)

# Create a symbolic link for the store-sales-time-series-forecasting competition data
symlink_target_comp = input_base_path / 'store-sales-time-series-forecasting'
symlink_source_comp = store_sales_time_series_forecasting_path

if symlink_target_comp.exists() or symlink_target_comp.is_symlink():
    if symlink_target_comp.is_symlink():
        os.unlink(symlink_target_comp)
    else:
        # If it's a directory, remove it to replace with symlink
        import shutil
        shutil.rmtree(symlink_target_comp)

os.symlink(symlink_source_comp, symlink_target_comp)
print(f"Created symlink: {symlink_target_comp} -> {symlink_source_comp}")

# Create a symbolic link for the ryanholbrook/ts-course-data dataset
symlink_target_data = input_base_path / 'ts-course-data'
symlink_source_data = ryanholbrook_ts_course_data_path

if symlink_target_data.exists() or symlink_target_data.is_symlink():
    if symlink_target_data.is_symlink():
        os.unlink(symlink_target_data)
    else:
        import shutil
        shutil.rmtree(symlink_target_data)

os.symlink(symlink_source_data, symlink_target_data)
print(f"Created symlink: {symlink_target_data} -> {symlink_source_data}")


In [ ]:
!git clone https://github.com/Kaggle/learntools.git
!mv learntools learntools_dir
!mv learntools_dir/learntools learntools

# 추세(Trend)란? #

시계열의 **추세** 구성 요소는 계열 평균의 지속적이고 장기적인 변화를 나타냅니다. 추세는 시계열에서 가장 느리게 움직이는 부분으로, 우리가 주목해야 할 가장 큰 시간 규모를 대표합니다. 제품 판매 시계열에서는 해마다 더 많은 사람이 제품을 알게 되면서 일어나는 시장 확대가 증가 추세로 나타날 수 있습니다.

<figure style="padding: 1em;">
<img src="https://storage.googleapis.com/kaggle-media/learn/images/ZdS4ZoJ.png" width=800, alt="">
<figcaption style="textalign: center; font-style: italic"><center>네 개의 시계열에서 나타나는 추세 패턴.</center></figcaption>
</figure>

이 강의에서는 평균의 추세에 집중하겠지만, 더 넓게 보면 느리고 지속적인 변화라면 무엇이든 추세라고 부를 수 있습니다. 예를 들어 시계열의 변동 폭에도 추세가 존재하는 경우가 흔합니다.

# 이동 평균 플롯 #

시계열 데이터가 어떤 흐름을 보이는지 알기 위해 이동 평균 그래프를 사용할 수 있습니다. 이동 평균을 구하려면 '슬라이딩 윈도우'라고 부르는 일정 구간을 정하고, 그 구간 안에 있는 값들의 평균을 냅니다. 그래프에 찍히는 각 파란선은 윈도우 안의 빨간 점들을 모두 더해 평균을 낸 결과입니다. 이렇게 하면 짧은 기간 동안 오르내리는 들쑥날쑥한 움직임은 부드러워지고, 긴 기간에 걸친 큰 흐름만 남게 됩니다.

<figure style="padding: 1em;">
<img src="https://storage.googleapis.com/kaggle-media/learn/images/EZOXiPs.gif" width=800, alt="An animated plot showing an undulating curve slowly increasing with a moving average line developing from left to right within a window of 12 points (in red).">
<figcaption style="textalign: center; font-style: italic"><center>선형 추세를 보여주는 이동 평균 그래프입니다. 파란색 선 위의 각 점은 크기가 12인 빨간색 윈도우 안에 들어있는 점들의 평균입니다.
</center></figcaption>
</figure>

위의 Mauna Loa 데이터는 해마다 반복해서 오르고 내리는 단기적인 계절성 변화를 가지고 있습니다. 어떤 변화가 '추세'라고 불리려면 이런 계절성 변화보다 더 긴 기간 동안 나타나야 합니다. 그래서 추세를 눈으로 확인하려면 계절의 길이보다 더 긴 기간을 잡아 평균을 내야 합니다. 여기서는 1년 동안의 계절 변화를 부드럽게 만들기 위해 윈도우 크기를 12로 잡았습니다.

# 추세 엔지니어링 #

추세의 형태를 파악했다면, '타임 스텝(time-step)' 피처를 활용해 모델링을 시도할 수 있습니다. 앞서 우리는 '타임 더미(time dummy)' 하나만 사용해도 선형 추세를 모델링할 수 있다는 사실을 확인했습니다.

```
target = a * time + b
```

이 타임 더미를 변형하면 훨씬 다양한 형태의 추세에 적합(fit)시킬 수 있습니다. 만약 추세가 포물선(2차 함수) 모양처럼 보인다면, 피처 집합에 '타임 더미의 제곱'을 추가하면 됩니다.
```
target = a * time ** 2 + b * time + c
```
선형 회귀는 `a`, `b`, `c` 계수를 학습합니다.

아래 그림의 추세 곡선은 이러한 피처와 scikit-learn의 `LinearRegression`으로 적합한 결과입니다.

<figure style="padding: 1em;">
<img src="https://storage.googleapis.com/kaggle-media/learn/images/KFYlgGm.png" width=*00, alt="Above, Cars Sold in Quebec: an undulating plot gradually increasing from 1960-01 to 1968-12 with a linear trend-line superimposed. Below, Plastics Production in Australia: an undulating plot with a concave-up quadratic trend-line superimposed.">
<figcaption style="textalign: center; font-style: italic"><center><strong>위:</strong> 선형 추세가 있는 시계열. <strong>아래:</strong> 이차 추세가 있는 시계열.
</center></figcaption>
</figure>

이 기법(Trick)을 처음 접했다면, 선형 회귀가 직선뿐만 아니라 곡선도 적합시킬 수 있다는 사실을 미처 몰랐을 수도 있습니다. 핵심 원리는 간단합니다. 적절한 형태의 곡선을 피처로 제공하기만 하면, 선형 회귀 모델이 타깃 데이터에 가장 잘 맞도록(Best fit) 이들을 결합하는 방법을 학습한다는 것입니다.

# 예제 - 터널 교통량 #

이번 예제에서는 *Tunnel Traffic* 데이터셋에 대한 추세 모델을 만들어 보겠습니다.


In [ ]:
from pathlib import Path
from warnings import simplefilter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

simplefilter("ignore")  # 경고를 무시해 출력 셀을 깔끔하게 유지

# Matplotlib 기본 설정
plt.style.use("seaborn-v0_8-whitegrid")
plt.rc("figure", autolayout=True, figsize=(11, 5))
plt.rc(
    "axes",
    labelweight="bold",
    labelsize="large",
    titleweight="bold",
    titlesize=14,
    titlepad=10,
)
plot_params = dict(
    color="0.75",
    style=".-",
    markeredgecolor="0.25",
    markerfacecolor="0.25",
    legend=False,
)
%config InlineBackend.figure_format = 'retina'


# 터널 교통량 데이터 불러오기
data_dir = Path("../input/ts-course-data")
tunnel = pd.read_csv(data_dir / "tunnel.csv", parse_dates=["Day"])
tunnel = tunnel.set_index("Day").to_period()

이 시리즈가 어떤 추세를 갖는지 확인하기 위해 이동 평균 플롯을 만들어 봅시다. 데이터가 일 단위 관측이므로 연도 안의 단기 변화를 부드럽게 만들도록 창 크기 365일을 선택하겠습니다.

이동 평균을 만들려면 먼저 `rolling` 메서드로 윈도우 연산을 시작한 뒤 `mean`을 호출해 윈도우 안의 평균을 계산합니다. 결과를 보면 *Tunnel Traffic*의 추세는 대략 선형인 것처럼 보입니다.


In [ ]:
moving_average = tunnel.rolling(
    window=365,       # 365일짜리 윈도우
    center=True,      # 평균을 윈도우의 중앙에 배치
    min_periods=183,  # 윈도우 길이의 절반 정도 사용
).mean()              # 평균을 계산(중앙값, 표준편차, 최솟값, 최댓값 등으로 바꿀 수도 있음)

ax = tunnel.plot(style=".", color="0.5")
moving_average.plot(
    ax=ax, linewidth=3, title="Tunnel Traffic - 365-Day Moving Average", legend=False,
);


1강에서는 판다스로 직접 타임 더미를 만들었습니다. 이제부터는 `statsmodels` 라이브러리의 `DeterministicProcess` 함수를 사용할 것입니다. 이 함수를 사용하면 시계열과 선형 회귀에서 발생할 수 있는 까다로운 실패 사례를 피할 수 있습니다. `order` 인수는 다항식 차수를 의미하며 `1`은 선형, `2`는 이차, `3`은 삼차를 뜻합니다.


In [ ]:
from statsmodels.tsa.deterministic import DeterministicProcess

dp = DeterministicProcess(
    index=tunnel.index,  # 학습 데이터의 날짜 인덱스
    constant=True,       # 편향(bias, y절편)을 위한 더미 피처
    order=1,             # 타임 더미(추세)
    drop=True,           # 다중공선성을 피하기 위해 필요하면 항을 제거
)
# `in_sample`은 `index` 인수로 전달한 날짜에 대한 피처를 생성
X = dp.in_sample()

X.head()


(참고로 *결정적 과정(deterministic process)*은 `const`와 `trend` 시리즈처럼 무작위성이 없거나 완전히 *결정된* 시계열을 말합니다. 시간 인덱스에서 파생한 피처는 일반적으로 결정적입니다.)

추세 모델을 만드는 절차는 이전과 거의 같지만, `fit_intercept=False` 인수를 추가한 점에 주의하세요.


In [ ]:
from sklearn.linear_model import LinearRegression

y = tunnel["NumVehicles"]  # 타깃

# 절편은 DeterministicProcess의 `const` 피처와 동일합니다.
# LinearRegression은 중복된 피처가 있으면 잘 동작하지 않으므로
# 여기서는 반드시 절편을 제외해야 합니다.
model = LinearRegression(fit_intercept=False)
model.fit(X, y)

y_pred = pd.Series(model.predict(X), index=X.index)


`LinearRegression` 모델이 찾아낸 추세는 이동 평균 플롯과 거의 동일합니다. 이 경우 선형 추세를 선택한 결정이 옳았다는 뜻이죠.


In [ ]:

ax = tunnel.plot(style=".", color="0.5", title="Tunnel Traffic - Linear Trend")
_ = y_pred.plot(ax=ax, linewidth=3, label="Trend")

예측을 만들려면 모델을 관측 기간 밖의 피처에 적용하면 됩니다. 여기서 “샘플 밖(out of sample)”은 학습 데이터가 관측된 기간을 벗어난 시간을 뜻합니다. 아래는 30일 예측을 만드는 방법의 한 예입니다.


In [ ]:
X = dp.out_of_sample(steps=30)

y_fore = pd.Series(model.predict(X), index=X.index)

y_fore.head()

다음 30일에 대한 추세 예측을 확인하기 위해 시리즈의 일부를 시각화해 봅시다.


In [ ]:

ax = tunnel["2005-05":].plot(title="Tunnel Traffic - Linear Trend Forecast", **plot_params)
ax = y_pred["2005-05":].plot(ax=ax, linewidth=3, label="Trend")
ax = y_fore.plot(ax=ax, linewidth=3, label="Trend Forecast", color="C3")
_ = ax.legend()

---

이번 강의에서 배운 추세 모델은 여러 이유로 유용합니다. 보다 정교한 모델을 위한 기준선이나 시작점이 될 뿐 아니라, 추세를 학습하지 못하는 알고리즘(XGBoost나 랜덤 포레스트 등)과 결합해 “하이브리드 모델”의 한 구성 요소로 사용할 수도 있습니다. 이 기법은 3일차에서 더 자세히 알아보겠습니다.

# 연습문제를 풀어보시죠 #


In [ ]:
# 피드백 시스템 설정
from learntools.core import binder
binder.bind(globals())
from learntools.time_series.ex2 import *

# 노트북 환경 설정
from pathlib import Path
from learntools.time_series.style import *  # 플롯 스타일 설정

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.linear_model import LinearRegression

data_dir = Path('../input/ts-course-data/')
comp_dir = Path('../input/store-sales-time-series-forecasting')

retail_sales = pd.read_csv(
    data_dir / "us-retail-sales.csv",
    parse_dates=['Month'],
    index_col='Month',
).to_period('D')
food_sales = retail_sales.loc[:, 'FoodAndBeverage']
auto_sales = retail_sales.loc[:, 'Automobiles']

dtype = {
    'store_nbr': 'category',
    'family': 'category',
    'sales': 'float32',
    'onpromotion': 'uint64',
}
store_sales = pd.read_csv(
    comp_dir / 'train.csv',
    dtype=dtype,
    parse_dates=['date'],
    infer_datetime_format=True,
)
store_sales = store_sales.set_index('date').to_period('D')
store_sales = store_sales.set_index(['store_nbr', 'family'], append=True)
average_sales = store_sales.groupby('date').mean()['sales']



# 1) 이동 평균 플롯으로 추세 파악하기

*US Retail Sales* 데이터셋에는 미국 여러 소매 업종의 월별 매출 데이터가 포함되어 있습니다. 다음 셀을 실행해 *Food and Beverage* 시리즈를 살펴보세요.


In [ ]:
ax = food_sales.plot(**plot_params)
ax.set(title="US Food and Beverage Sales", ylabel="Millions of Dollars");

이제 이동 평균 플롯을 만들어 이 시리즈의 추세를 추정해 보세요.


In [ ]:
# 직접 작성하세요: 추세를 추정할 수 있도록 `food_sales`에 적절한 매개변수의 이동평균을
# 계산하는 메서드를 직접 추가하세요.
trend = food_sales._____


# 플롯 만들기
ax = food_sales.plot(**plot_params, alpha=0.5)
ax = trend.plot(ax=ax, linewidth=3)


In [ ]:
# 힌트나 해설이 필요하면 아래 주석을 해제하세요
#q_1.hint()
#q_1.solution()


-------------------------------------------------------------------------------

# 2) 추세 식별하기

*Food and Beverage Sales* 시리즈에는 몇 차 다항식 추세가 적당할까요? 더 잘 맞을 것 같은 비다항식 곡선이 떠오르나요?

생각을 정리한 뒤 이 셀을 실행해 해설을 확인하세요.


In [ ]:
# 해설 보기(이 셀을 실행해야 점수를 받을 수 있습니다!)
q_2.check()


-------------------------------------------------------------------------------

이번 강의에서는 평균 매출 시계열을 계속 사용합니다. `average_sales`의 추세를 추정하는 이동 평균 플롯을 보려면 이 셀을 실행하세요.


In [ ]:
trend = average_sales.rolling(
    window=365,
    center=True,
    min_periods=183,
).mean()

ax = average_sales.plot(**plot_params, alpha=0.5)
ax = trend.plot(ax=ax, linewidth=3)

# 3) 추세 피처 만들기

`DeterministicProcess`를 사용해 3차(큐빅) 추세 모델용 피처 집합을 만드세요. 90일 예측에 사용할 피처도 생성하세요.


In [ ]:
from statsmodels.tsa.deterministic import DeterministicProcess

y = average_sales.copy()  # 타깃

# 직접 작성하세요: 3차 추세 모델에 맞는 인수로 `DeterministicProcess` 인스턴스를 만드세요
dp = ____

# 직접 작성하세요: y.index에 해당하는 날짜들의 피처 집합을 만드세요
X = ____

# 직접 작성하세요: 90일 예측을 위한 피처를 만드세요.
X_fore = ____




In [ ]:
# 아래 줄들은 힌트나 해설 코드를 제공합니다
#q_3.hint()
#q_3.solution()


다음 셀을 실행하면 결과 플롯을 확인할 수 있습니다.


In [ ]:
model = LinearRegression()
model.fit(X, y)

y_pred = pd.Series(model.predict(X), index=X.index)
y_fore = pd.Series(model.predict(X_fore), index=X_fore.index)

ax = y.plot(**plot_params, alpha=0.5, title="Average Sales", ylabel="items sold")
ax = y_pred.plot(ax=ax, linewidth=3, label="Trend", color='C0')
ax = y_fore.plot(ax=ax, linewidth=3, label="Trend Forecast", color='C3')
ax.legend();

--------------------------------------------------------------------------------

더 복잡한 추세를 맞추는 한 가지 방법은 사용하는 다항식의 차수를 높이는 것입니다. *Store Sales*의 다소 복잡한 추세에 더 잘 맞도록 11차 다항식을 시도해 볼 수 있습니다.


In [ ]:
from statsmodels.tsa.deterministic import DeterministicProcess

dp = DeterministicProcess(index=y.index, order=11)
X = dp.in_sample()

model = LinearRegression()
model.fit(X, y)

y_pred = pd.Series(model.predict(X), index=X.index)

ax = y.plot(**plot_params, alpha=0.5, title="Average Sales", ylabel="items sold")
ax = y_pred.plot(ax=ax, linewidth=3, label="Trend", color='C0')
ax.legend();

# 4) 고차 다항식으로 예측할 때의 위험 이해하기

그렇지만 고차 다항식은 일반적으로 예측 작업에 적합하지 않습니다. 이유가 무엇일지 추측해 보세요.


In [ ]:
# 해설 보기
q_4.check()


In [ ]:
# 힌트가 필요하면 다음 줄의 주석을 해제하세요
#q_4.hint()


90일 예측을 11차 다항식으로 다시 계산한 결과를 보려면 이 셀을 실행하세요. 여러분의 예상과 들어맞나요?


In [ ]:
X_fore = dp.out_of_sample(steps=90)
y_fore = pd.Series(model.predict(X_fore), index=X_fore.index)

ax = y.plot(**plot_params, alpha=0.5, title="Average Sales", ylabel="items sold")
ax = y_pred.plot(ax=ax, linewidth=3, label="Trend", color='C0')
ax = y_fore.plot(ax=ax, linewidth=3, label="Trend Forecast", color='C3')
ax.legend();